In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data_utils


import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

import numpy as np
import random

from sklearn.datasets import load_iris

# StandardScaler

from sklearn.preprocessing import StandardScaler
from torchmetrics.classification import MulticlassAccuracy

%matplotlib inline

In [2]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [3]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((120, 4), (120,), (30, 4), (30,))

In [4]:
classes = np.unique(y_train)

In [5]:
classes

array([0, 1, 2])

In [6]:
ss = StandardScaler()

X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((120, 4), (120,), (30, 4), (30,))

In [7]:
np.mean(X_train, axis=0), np.std(X_train, axis=0), np.min(X_train, axis=0), np.max(X_train, axis=0)

(array([ 3.51570624e-15,  3.06467814e-16, -3.96904731e-16,  3.69149156e-16]),
 array([1., 1., 1., 1.]),
 array([-1.7076718 , -2.51579615, -1.5618717 , -1.44726656]),
 array([2.39777844, 2.4879973 , 1.75289198, 1.69849019]))

In [8]:
# Convert to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [9]:
# Convert to DataLoader

train_ds = data_utils.TensorDataset(X_train, y_train)
test_ds = data_utils.TensorDataset(X_test, y_test)

In [10]:
train_dl = data_utils.DataLoader(train_ds, batch_size=16, shuffle=True)
test_dl = data_utils.DataLoader(test_ds, batch_size=16, shuffle=False)

In [11]:
for i, data in enumerate(train_dl):
    print(f"Data: {data[0]}, Label: {data[1]}, Index: {i}")

Data: tensor([[ 0.0518, -0.1330,  0.7416,  0.7810],
        [ 1.8113, -0.6096,  1.3034,  0.9121],
        [ 0.5210, -0.3713,  1.0225,  0.7810],
        [-0.0655,  2.2497, -1.4495, -1.3162],
        [ 0.8729, -0.3713,  0.4607,  0.1256],
        [ 0.7556,  0.3435,  0.7416,  1.0431],
        [-1.1212,  1.2966, -1.3371, -1.4473],
        [ 2.1632, -1.0861,  1.7529,  1.4363],
        [ 1.6940, -0.3713,  1.4158,  0.7810],
        [-1.4731,  0.1052, -1.2810, -1.3162],
        [ 1.1075, -0.6096,  0.5731,  0.2567],
        [-1.2385, -0.1330, -1.3371, -1.4473],
        [-1.5904, -1.8010, -1.3933, -1.1851],
        [-1.0039, -2.5158, -0.1573, -0.2676],
        [ 0.2864, -0.6096,  0.5169, -0.0055],
        [ 0.6383, -0.6096,  1.0225,  1.3053]]), Label: tensor([2, 2, 2, 0, 1, 2, 0, 2, 2, 0, 1, 0, 0, 1, 1, 2]), Index: 0
Data: tensor([[-1.1212,  0.1052, -1.2810, -1.3162],
        [-1.0039,  0.8201, -1.2248, -1.0540],
        [ 0.9902, -0.1330,  0.6854,  0.6499],
        [-1.0039, -0.1330, -1.2248, -1

In [12]:
class MLP(nn.Module):
    
    def __init__(self):
        super().__init__()

        self.layer_1 = nn.Linear(in_features=4, out_features=16)
        self.layer_2 = nn.Dropout(p=0.2)
        self.layer_3 = nn.Linear(in_features=16, out_features=3)
        

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = F.relu(self.layer_2(x))
        x = self.layer_3(x)

        return x

In [13]:
model = MLP()

In [14]:
for i in model.parameters():
    print(i.shape, i.dtype, i.device)

torch.Size([16, 4]) torch.float32 cpu
torch.Size([16]) torch.float32 cpu
torch.Size([3, 16]) torch.float32 cpu
torch.Size([3]) torch.float32 cpu


In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)

#Accuracy
accuracy_fn_train = MulticlassAccuracy(num_classes=len(classes), average='micro')
accuracy_fn_test = MulticlassAccuracy(num_classes=len(classes), average='micro')

In [16]:
epochs = 50

train_loss_values = []
test_loss_values = []
epoch_count = []
accuracy_values = []

for epoch in range(epochs):
    model.train()

    # Metric reset
    loss_epoch = 0

    accuracy_fn_train.reset()

    for i, data in enumerate(train_dl, 0):

      X = data[0]
      y = data[1]

      y_pred = model(X)

      # Loss 
      loss = loss_fn(y_pred, y)
      loss_epoch += loss

      # Accuracy
      accuracy_fn_train.update(y_pred.argmax(dim=1), y)

      # 3. Azzeramento dei gradienti
      optimizer.zero_grad()

      # 4. Backpropagation
      loss.backward()

      # 5. Ottimizzazione
      optimizer.step()

    accuracy_train = accuracy_fn_train.compute()

    #reset loss
    loss_test = 0

    #Eval phase
    model.eval()

    accuracy_fn_test.reset()
    
    for j, data in enumerate(test_dl, 0):

      X = data[0] #[128, 8]
      y = data[1] #[128 ,1]

      accuracy_epoch = 0

      total = 0
      correct = 0

      with torch.no_grad():

        # 1. Forward pass
        y_pred = model(X) #[128 ,1]
        # 2. Calcolo della loss
        loss = loss_fn(y_pred, y)
        loss_test += loss

        # Manual Accuracy
        _, predicted = torch.max(y_pred, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

        # Accuracy
        #print(f"Predictions: {y_pred.argmax(dim=1)}, Labels: {y}")
        accuracy_fn_test.update(y_pred.argmax(dim=1), y)

    accuracy_test = accuracy_fn_test.compute()

    accuracy_test_manual = correct / total
    # print(f"Accuracy Test Manual: {accuracy_test_manual:.3f}")
    # print(f"Accuracy Test: {accuracy_test:.3f}")

    epoch_count.append(epoch)
    train_loss_values.append(loss_epoch.detach().numpy()/len(train_dl))
    test_loss_values.append(loss_test.detach().numpy()/len(test_dl))
    accuracy_values.append(accuracy_test)

    print(f"Epoca: {epoch} |  Train Loss: {loss_epoch/len(train_dl):.3f} | Test Loss: {loss_test/len(test_dl):.3f} | Train Accuracy: {accuracy_train:.3f} | Test Accuracy: {accuracy_test:.3f}")

Epoca: 0 |  Train Loss: 1.088 | Test Loss: 0.902 | Train Accuracy: 0.375 | Test Accuracy: 0.667
Epoca: 1 |  Train Loss: 0.828 | Test Loss: 0.667 | Train Accuracy: 0.742 | Test Accuracy: 0.933
Epoca: 2 |  Train Loss: 0.612 | Test Loss: 0.483 | Train Accuracy: 0.775 | Test Accuracy: 0.933
Epoca: 3 |  Train Loss: 0.442 | Test Loss: 0.381 | Train Accuracy: 0.825 | Test Accuracy: 0.933
Epoca: 4 |  Train Loss: 0.410 | Test Loss: 0.327 | Train Accuracy: 0.800 | Test Accuracy: 0.900
Epoca: 5 |  Train Loss: 0.333 | Test Loss: 0.309 | Train Accuracy: 0.858 | Test Accuracy: 0.867
Epoca: 6 |  Train Loss: 0.302 | Test Loss: 0.253 | Train Accuracy: 0.850 | Test Accuracy: 0.933
Epoca: 7 |  Train Loss: 0.309 | Test Loss: 0.215 | Train Accuracy: 0.883 | Test Accuracy: 0.967
Epoca: 8 |  Train Loss: 0.281 | Test Loss: 0.188 | Train Accuracy: 0.875 | Test Accuracy: 0.967
Epoca: 9 |  Train Loss: 0.254 | Test Loss: 0.161 | Train Accuracy: 0.908 | Test Accuracy: 0.967
Epoca: 10 |  Train Loss: 0.242 | Test Lo